In [1]:
# Phase 1 — Data Generation & Cleaning
# Project: Savoria Restaurant Group — Financial Analysis
# Author: Leo SC

# ── Libraries ──────────────────────────────────────────
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import warnings
warnings.filterwarnings('ignore')

# Set seed for reproducibility
np.random.seed(42)
random.seed(42)

print("Libraries loaded ✓")
print(f"Project: Savoria Restaurant Group")
print(f"Analysis period: 2022 - 2024")

Libraries loaded ✓
Project: Savoria Restaurant Group
Analysis period: 2022 - 2024


In [2]:
# ── Business Structure ──────────────────────────────────
branches = {
    'BR001': {'name': 'Downtown',    'city': 'Mexico City',   'opened': '2020-01-15', 'size': 'large'},
    'BR002': {'name': 'Polanco',     'city': 'Mexico City',   'opened': '2020-06-01', 'size': 'large'},
    'BR003': {'name': 'Guadalajara', 'city': 'Guadalajara',   'opened': '2021-03-10', 'size': 'medium'},
    'BR004': {'name': 'Monterrey',   'city': 'Monterrey',     'opened': '2021-09-20', 'size': 'medium'},
    'BR005': {'name': 'Reforma',     'city': 'Mexico City',   'opened': '2022-02-14', 'size': 'small'},
}

menu_categories = {
    'Starters':    {'avg_price': 85,  'food_cost_pct': 0.28},
    'Main Course': {'avg_price': 180, 'food_cost_pct': 0.32},
    'Desserts':    {'avg_price': 75,  'food_cost_pct': 0.25},
    'Beverages':   {'avg_price': 60,  'food_cost_pct': 0.18},
    'Alcohol':     {'avg_price': 120, 'food_cost_pct': 0.22},
}

capacity = {'large': 120, 'medium': 80, 'small': 50}

print("Business structure defined ✓")
print(f"\nBranches: {len(branches)}")
print(f"Menu categories: {len(menu_categories)}")
for bid, info in branches.items():
    print(f"  {bid} — {info['name']:<12} ({info['city']:<15}) Size: {info['size']}")

Business structure defined ✓

Branches: 5
Menu categories: 5
  BR001 — Downtown     (Mexico City    ) Size: large
  BR002 — Polanco      (Mexico City    ) Size: large
  BR003 — Guadalajara  (Guadalajara    ) Size: medium
  BR004 — Monterrey    (Monterrey      ) Size: medium
  BR005 — Reforma      (Mexico City    ) Size: small


In [3]:
# ── Generate Daily Sales Data ───────────────────────────
def generate_sales_data(branches, menu_categories, capacity):
    
    records = []
    date_range = pd.date_range(start='2022-01-01', end='2024-12-31', freq='D')
    
    for date in date_range:
        for branch_id, branch in branches.items():
            
            # Skip days before branch opened
            if date < pd.Timestamp(branch['opened']):
                continue
            
            # Base occupancy by day of week
            is_weekend   = date.dayofweek >= 5
            is_friday    = date.dayofweek == 4
            base_occ     = 0.75 if is_weekend else (0.65 if is_friday else 0.50)
            
            # Seasonality — higher in Dec, lower in Jan/Feb
            month_factor = {1: 0.85, 2: 0.80, 3: 0.90, 4: 0.95,
                           5: 1.00, 6: 1.05, 7: 1.10, 8: 1.05,
                           9: 0.95, 10: 1.00, 11: 1.05, 12: 1.20}[date.month]
            
            # Branch size factor
            size_factor = {'large': 1.0, 'medium': 0.75, 'small': 0.50}[branch['size']]
            
            # Growth trend year over year
            year_growth = {2022: 1.0, 2023: 1.12, 2024: 1.22}[date.year]
            
            # Calculate covers (number of diners)
            max_cap   = capacity[branch['size']]
            occupancy = base_occ * month_factor * year_growth
            occupancy = min(occupancy, 1.0)  # Cap at 100%
            covers    = int(max_cap * occupancy * np.random.uniform(0.85, 1.15))
            
            # Generate sales by menu category
            for category, cat_info in menu_categories.items():
                
                # Category mix — not all diners order everything
                cat_mix = {
                    'Starters':    0.60,
                    'Main Course': 0.95,
                    'Desserts':    0.40,
                    'Beverages':   0.85,
                    'Alcohol':     0.45,
                }[category]
                
                orders      = int(covers * cat_mix * np.random.uniform(0.90, 1.10))
                avg_price   = cat_info['avg_price'] * np.random.uniform(0.92, 1.08)
                revenue     = orders * avg_price
                food_cost   = revenue * cat_info['food_cost_pct'] * np.random.uniform(0.95, 1.05)
                
                records.append({
                    'date':        date,
                    'branch_id':   branch_id,
                    'branch_name': branch['name'],
                    'city':        branch['city'],
                    'category':    category,
                    'covers':      covers,
                    'orders':      orders,
                    'revenue':     round(revenue, 2),
                    'food_cost':   round(food_cost, 2),
                    'is_weekend':  is_weekend,
                    'month':       date.month,
                    'year':        date.year,
                    'quarter':     date.quarter,
                })
    
    return pd.DataFrame(records)

df_sales = generate_sales_data(branches, menu_categories, capacity)

print(f"Records generated: {len(df_sales):,}")
print(f"Date range: {df_sales['date'].min().date()} → {df_sales['date'].max().date()}")
print(f"Branches: {df_sales['branch_id'].nunique()}")
print(f"\nSample data:")
print(df_sales.head(3).to_string())

Records generated: 27,180
Date range: 2022-01-01 → 2024-12-31
Branches: 5

Sample data:
        date branch_id branch_name         city     category  covers  orders   revenue  food_cost  is_weekend  month  year  quarter
0 2022-01-01     BR001    Downtown  Mexico City     Starters      73      47   4143.29    1171.57        True      1  2022        1
1 2022-01-01     BR001    Downtown  Mexico City  Main Course      73      64  10885.93    3329.56        True      1  2022        1
2 2022-01-01     BR001    Downtown  Mexico City     Desserts      73      31   2362.61     602.94        True      1  2022        1


In [4]:
# ── Add Realistic Noise & Errors ────────────────────────
df_noisy = df_sales.copy()

# 1. Missing values — simulates data entry gaps (1.5% of records)
missing_idx = df_noisy.sample(frac=0.015).index
df_noisy.loc[missing_idx, 'revenue'] = np.nan

# 2. Outliers — unusually high sales (special events, catering)
outlier_idx = df_noisy.sample(frac=0.005).index
df_noisy.loc[outlier_idx, 'revenue'] *= np.random.uniform(2.5, 4.0)

# 3. Negative values — data entry errors
error_idx = df_noisy.sample(frac=0.002).index
df_noisy.loc[error_idx, 'revenue'] *= -1

# 4. Duplicate records — system glitches
duplicates = df_noisy.sample(frac=0.003)
df_noisy = pd.concat([df_noisy, duplicates], ignore_index=True)

print("Noise added ✓")
print(f"\nData quality issues introduced:")
print(f"  Missing values:   {df_noisy['revenue'].isna().sum():>6} records")
print(f"  Negative values:  {(df_noisy['revenue'] < 0).sum():>6} records")
print(f"  Duplicate rows:   {df_noisy.duplicated().sum():>6} records")
print(f"  Total records:    {len(df_noisy):,}")

Noise added ✓

Data quality issues introduced:
  Missing values:      409 records
  Negative values:      53 records
  Duplicate rows:       82 records
  Total records:    27,262


In [5]:
# ── Data Cleaning Pipeline ──────────────────────────────
df_clean = df_noisy.copy()
issues_log = {}

# Step 1 — Remove duplicates
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
issues_log['duplicates_removed'] = before - len(df_clean)

# Step 2 — Remove negative values
mask_negative = df_clean['revenue'] < 0
issues_log['negatives_removed'] = mask_negative.sum()
df_clean = df_clean[~mask_negative]

# Step 3 — Flag and fill outliers
q99 = df_clean['revenue'].quantile(0.99)
mask_outlier = df_clean['revenue'] > q99
issues_log['outliers_flagged'] = mask_outlier.sum()
df_clean['is_outlier'] = mask_outlier
df_clean.loc[mask_outlier, 'revenue'] = df_clean.loc[mask_outlier, 'revenue'].clip(upper=q99)

# Step 4 — Impute missing values with branch+category median
df_clean['revenue'] = df_clean.groupby(
    ['branch_id', 'category']
)['revenue'].transform(lambda x: x.fillna(x.median()))
issues_log['missing_imputed'] = df_noisy['revenue'].isna().sum()

# Step 5 — Recalculate food_cost where revenue was imputed
mask_recalc = df_clean['food_cost'].isna()
for category, cat_info in menu_categories.items():
    cat_mask = mask_recalc & (df_clean['category'] == category)
    df_clean.loc[cat_mask, 'food_cost'] = (
        df_clean.loc[cat_mask, 'revenue'] * cat_info['food_cost_pct']
    )

# Step 6 — Add gross profit column
df_clean['gross_profit'] = df_clean['revenue'] - df_clean['food_cost']
df_clean['gross_margin']  = df_clean['gross_profit'] / df_clean['revenue']

print("═" * 50)
print("DATA CLEANING REPORT")
print("═" * 50)
print(f"  Original records:    {len(df_noisy):>8,}")
print(f"  Duplicates removed:  {issues_log['duplicates_removed']:>8,}")
print(f"  Negatives removed:   {issues_log['negatives_removed']:>8,}")
print(f"  Outliers flagged:    {issues_log['outliers_flagged']:>8,}")
print(f"  Missing imputed:     {issues_log['missing_imputed']:>8,}")
print(f"  Clean records:       {len(df_clean):>8,}")
print(f"\n  Revenue range:  ${df_clean['revenue'].min():,.0f} — ${df_clean['revenue'].max():,.0f}")
print(f"  Avg gross margin:    {df_clean['gross_margin'].mean():.1%}")
print("═" * 50)

══════════════════════════════════════════════════
DATA CLEANING REPORT
══════════════════════════════════════════════════
  Original records:      27,262
  Duplicates removed:        82
  Negatives removed:         53
  Outliers flagged:         268
  Missing imputed:          409
  Clean records:         27,127

  Revenue range:  $443 — $18,090
  Avg gross margin:    75.0%
══════════════════════════════════════════════════


In [6]:
# ── Save Clean Data ─────────────────────────────────────

# Save as CSV for SQL and Power BI
df_clean.to_csv('data_clean.csv', index=False)

# Save as Excel for financial modeling
df_clean.to_excel('data_clean.xlsx', index=False)

# Quick validation
print("Files saved ✓")
print(f"\nFinal dataset summary:")
print(f"  Records:    {len(df_clean):,}")
print(f"  Columns:    {len(df_clean.columns)}")
print(f"  Date range: {df_clean['date'].min().date()} → {df_clean['date'].max().date()}")
print(f"  Branches:   {df_clean['branch_id'].nunique()}")
print(f"\nColumns:")
for col in df_clean.columns:
    print(f"  • {col}")

Files saved ✓

Final dataset summary:
  Records:    27,127
  Columns:    16
  Date range: 2022-01-01 → 2024-12-31
  Branches:   5

Columns:
  • date
  • branch_id
  • branch_name
  • city
  • category
  • covers
  • orders
  • revenue
  • food_cost
  • is_weekend
  • month
  • year
  • quarter
  • is_outlier
  • gross_profit
  • gross_margin


## Phase 1 — Data Generation & Cleaning

### Business Context
Savoria Restaurant Group — 5 branches across Mexico (2022-2024)

### Data Generated
- 27,262 raw records across 5 branches and 5 menu categories
- 3 years of daily sales data with realistic seasonality and growth trends

### Data Quality Issues Found & Resolved
| Issue | Records |
|-------|---------|
| Duplicates removed | 82 |
| Negative values removed | 53 |
| Outliers flagged & capped | 268 |
| Missing values imputed | 409 |
| **Clean records** | **27,127** |

### Key Metrics
- Avg Gross Margin: 75%
- Revenue range: $443 — $18,090 per record
- Growth trend: +12% (2023) and +22% (2024) vs 2022

### Output Files
- `data_clean.csv` — for SQL and Power BI
- `data_clean.xlsx` — for financial modeling